<a href="https://colab.research.google.com/github/WoochangShin98/Economic-activity-from-nighttime-imagery/blob/main/statistic_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
BASE_DIR = "/content/drive/MyDrive/CSCI 5527 Deep Learning/Project"

IMAGE_DIR = os.path.join(BASE_DIR, "opt1_county_only")
LABEL_CSV = os.path.join(BASE_DIR, "all_state_county_poverty.csv")

FEATURE_CSV = os.path.join(BASE_DIR, "png_features.csv")
FINAL_CSV = os.path.join(BASE_DIR, "final_stat_dataset_no_shapefile.csv")

print("IMAGE_DIR exists:", os.path.exists(IMAGE_DIR))
print("LABEL_CSV exists:", os.path.exists(LABEL_CSV))

IMAGE_DIR exists: True
LABEL_CSV exists: True


In [15]:
def extract_fips(filename):
    parts = filename.split("_")
    if len(parts) >= 2:
        return parts[1].zfill(5)
    return None

In [16]:
def extract_png_features(image_path):
    img = Image.open(image_path).convert("RGBA")
    arr = np.array(img)

    rgb = arr[:, :, :3]
    alpha = arr[:, :, 3]

    # county 內部：非透明
    mask = alpha > 0

    if mask.sum() == 0:
        return None

    gray = (
        0.299 * rgb[:, :, 0]
        + 0.587 * rgb[:, :, 1]
        + 0.114 * rgb[:, :, 2]
    )

    pixels = gray[mask].astype(float)
    pixels = pixels[np.isfinite(pixels)]

    if len(pixels) == 0:
        return None

    features = {
        "mean_light": np.mean(pixels),
        "std_light": np.std(pixels),
        "max_light": np.max(pixels),
        "min_light": np.min(pixels),
        "median_light": np.median(pixels),
        "p25_light": np.percentile(pixels, 25),
        "p75_light": np.percentile(pixels, 75),
        "p90_light": np.percentile(pixels, 90),
        "p95_light": np.percentile(pixels, 95),
        "total_light": np.sum(pixels),
        "county_pixel_count": len(pixels),
        "lit_share_10": np.mean(pixels > 10),
        "lit_share_30": np.mean(pixels > 30),
        "lit_share_50": np.mean(pixels > 50),
        "lit_share_100": np.mean(pixels > 100),
    }

    return features

In [17]:
image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(".png")
]

print("Number of PNG files:", len(image_files))
print(image_files[:5])

Number of PNG files: 3218
['22_22039_Evangeline.png', '51_51650_Hampton.png', '20_20175_Seward.png', '46_46049_Faulk.png', '55_55085_Oneida.png']


In [18]:
rows = []

for filename in tqdm(image_files):
    fips = extract_fips(filename)

    if fips is None:
        continue

    image_path = os.path.join(IMAGE_DIR, filename)

    feats = extract_png_features(image_path)

    if feats is None:
        continue

    feats["ID"] = fips
    feats["filename"] = filename
    feats["image_path"] = image_path

    rows.append(feats)

features = pd.DataFrame(rows)

print(features.head())
print(features.shape)

features.to_csv(FEATURE_CSV, index=False)
print("Saved:", FEATURE_CSV)

 57%|█████▋    | 1843/3218 [00:48<00:13, 102.75it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (126859904 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
100%|██████████| 3218/3218 [01:06<00:00, 48.07it/s] 


   mean_light  std_light  max_light  min_light  median_light  p25_light  \
0   14.301629  23.627300      193.0        0.0           0.0        0.0   
1   76.163645  69.638068      254.0        0.0          70.0        0.0   
2    8.603205  25.624125      230.0        0.0           0.0        0.0   
3    1.917544   7.816790      141.0        0.0           0.0        0.0   
4    5.856690  17.293992      195.0        0.0           0.0        0.0   

   p75_light  p90_light  p95_light  total_light  county_pixel_count  \
0       28.0       39.0      51.00     141343.0                9883   
1      141.0      172.0     192.75     168017.0                2206   
2        0.0       30.0      44.00      84819.0                9859   
3        0.0        0.0      22.00      33511.0               17476   
4        0.0       25.0      33.00     127588.0               21785   

   lit_share_10  lit_share_30  lit_share_50  lit_share_100     ID  \
0      0.369422      0.200040      0.050997       0.0

In [19]:
labels = pd.read_csv(LABEL_CSV)

labels["ID"] = labels["ID"].astype(str).str.zfill(5)

labels = labels.rename(columns={
    "Percent in Poverty": "poverty_rate"
})

labels = labels[labels["Name"].str.contains("County", case=False, na=False)]

labels = labels[["ID", "Name", "poverty_rate"]]

print(labels.head())
print(labels.shape)

      ID            Name  poverty_rate
1  01001  Autauga County          12.9
2  01003  Baldwin County          10.5
3  01005  Barbour County          28.1
4  01007     Bibb County          21.6
5  01009   Blount County          11.7
(2994, 3)


In [20]:
features["ID"] = features["ID"].astype(str).str.zfill(5)

df = features.merge(labels, on="ID", how="inner")

print(df.head())
print("features:", features.shape)
print("labels:", labels.shape)
print("merged:", df.shape)

df.to_csv(FINAL_CSV, index=False)
print("Saved:", FINAL_CSV)

   mean_light  std_light  max_light  min_light  median_light  p25_light  \
0    8.603205  25.624125      230.0        0.0           0.0        0.0   
1    1.917544   7.816790      141.0        0.0           0.0        0.0   
2    5.856690  17.293992      195.0        0.0           0.0        0.0   
3   62.855317  43.536011      221.0        0.0          57.0       41.0   
4   11.851994  17.326268      170.0        0.0           0.0        0.0   

   p75_light  p90_light  p95_light  total_light  county_pixel_count  \
0        0.0       30.0       44.0      84819.0                9859   
1        0.0        0.0       22.0      33511.0               17476   
2        0.0       25.0       33.0     127588.0               21785   
3       81.0      126.0      155.0     152487.0                2426   
4       25.0       32.0       38.0      67746.0                5716   

   lit_share_10  lit_share_30  lit_share_50  lit_share_100     ID  \
0      0.178213      0.096359      0.042296       0.0

In [21]:
feature_cols = [
    "mean_light",
    "std_light",
    "max_light",
    "min_light",
    "median_light",
    "p25_light",
    "p75_light",
    "p90_light",
    "p95_light",
    "total_light",
    "county_pixel_count",
    "lit_share_10",
    "lit_share_30",
    "lit_share_50",
    "lit_share_100",
]

X = df[feature_cols].copy()
y = df["poverty_rate"].copy()

X = X.replace([np.inf, -np.inf], np.nan)
valid_idx = X.dropna().index

X = X.loc[valid_idx]
y = y.loc[valid_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [22]:
models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    print("\n" + "=" * 40)
    print(name)
    print("=" * 40)
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)

results_df = pd.DataFrame(results)
results_df


Linear Regression
MAE: 3.8029466694393865
RMSE: 4.944354459570629
R2: 0.08718902915294235

Random Forest
MAE: 3.8566198484061953
RMSE: 5.027702384114821
R2: 0.05615478366887916


,Model,MAE,RMSE,R2
0,Linear Regression,3.802947,4.944354,0.087189
1,Random Forest,3.856620,5.027702,0.056155


county PNG image
   ↓
extract brightness features
   ↓
merge poverty label
   ↓
Linear Regression / Random Forest
   ↓
predict poverty_rate

Linear Regression
MAE ≈ 3.80
R2 ≈ 0.087

Random Forest
MAE ≈ 3.86
R2 ≈ 0.056

主要原因 1：你現在預測的是 poverty，不是 GDP

這很重要。

夜間燈光比較直接反映的是：

人口密度
城市化程度
基礎建設
商業活動
GDP / economic activity

但 poverty rate 受到很多其他因素影響：

教育程度
產業結構
收入分配
生活成本
失業率
社會福利
家庭結構
地區政策

所以：

nighttime light → GDP 可能比較直接
nighttime light → poverty 比較間接

因此 R² 很低是合理的。

主要原因 2：你的 feature 還是太粗

你現在的 feature 是：

mean_light
std_light
max_light
p90_light
total_light
lit_share

這些都是「整體亮度統計」。

它們只能回答：

這個 county 整體亮不亮？

但不能回答：

這些光代表什麼？
人口多不多？
城市密不密？
是一個大城市，還是很多小村莊？
是工業區，還是住宅區？

所以資訊不夠。

主要原因 3：Random Forest 沒有變好，代表 feature 訊息不足

通常 Random Forest 可以抓非線性關係。

但你現在：

Linear Regression R2 ≈ 0.087
Random Forest R2 ≈ 0.056

Random Forest 反而更差。

這表示：

問題不是模型太簡單，而是 feature representation 本身沒有足夠 poverty signal。

這句可以直接寫進報告。

主要原因 4：PNG 可能不是原始 VIIRS radiance

如果你的 PNG 是從 tif 轉出來的顯示圖，它可能是 0–255 的影像值，不是原始 radiance。

這代表：

你現在的 feature 是 image brightness，不一定是 real nighttime light intensity。

如果要做最嚴謹的統計 baseline，最好用原始 .tif 抽 feature。